<a href="https://colab.research.google.com/github/ShamGaneshan2008/.ipynb-files-/blob/main/OMEGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **OMEGA**
an AI that learns how a driving environment evolves, imagines possible futures, estimates danger, and chooses safer actions.

In [ ]:
!pip install -q numpy pandas matplotlib seaborn scikit-learn
!pip install -q torch torchvision torchaudio
!pip install -q networkx tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import sklearn
import networkx as nx

In [ ]:
from pathlib import Path

folders = [
    "omega",
    "omega/data",
    "omega/models",
    "omega/world_model",
    "omega/uncertainty",
    "omega/counterfactual",
    "omega/memory",
    "omega/planner",
    "omega/utils",
    "experiments",
    "results"
]

for folder in folders:
  Path(folder).mkdir(parents=True, exist_ok=True)

print("OMEGA-SENTINEL structure created.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using: ", device)

if device.type == "cuda":
  print("GPU: ", torch.cuda.get_device_name(0))

In [ ]:
!pip uninstall -y tensorflow tensorflow-gpu
!pip install -q tensorflow==2.12.0
!pip install -q waymo-open-dataset-tf-2-12-0
import tensorflow as tf
from waymo_open_dataset import dataset_pb2

print("Tensorflow: ", tf.__version__)
print("Waymo API loaded successfully!")

In [ ]:
!wget -q --show-progress \
https://storage.googleapis.com/waymo-open-dataset/waymo-open-dataset/v_1_4_2/individual_files/training/training_0000.tfrecord \
-O /content/training_0000.tfrecord

In [ ]:
import os

file_path = "/content/training_0000.tfrecord"

print("File exists: ", os.path.exists(file_path))
print("Size: ", os.path.getsize(file_path) / (1024**3), "GB")

In [ ]:
import tensorflow as tf

dataset = tf.data.TFRecordDataset(
    file_path,
    compression_type=""
)

for raw_record in dataset.take(1):
  print("Record loaded!")
  print("Byte: ", len(raw_record.numpy()))

In [ ]:
from waymo_open_dataset.protos import scenario_pb2

scenario = scenario_pb2.Scenario()

for raw_record in dataset.take(1):
  scenario.ParseFromString(raw_record.numpy())

print("Scenario ID: ", scenario.scenario_id)
print("Number of tracks: ", len(scenario.tracks))
print("Number of timestamps:", len(scenario.timestamps_seconds))

for i, track in enumerate(scenario.tracks[:10]):
    print(
        "Track:", i,
        "| ID:", track.id,
        "| type:", track.object_type
    )

In [ ]:
trajectories = []

for track in scenario.tracks:

    states = []

    for state in track.states:

        states.append([
            state.center_x,
            state.center_y,
            state.length,
            state.width,
            state.heading,
            state.velocity_x,
            state.velocity_y
        ])

    trajectories.append({
        "track_id": track.id,
        "object_type": track.object_type,
        "states": np.array(states)
    })

print("Objects extracted:", len(trajectories))

In [ ]:
vehicle = trajectories[0]

print("Track ID:", vehicle["track_id"])
print("Object type:", vehicle["object_type"])
print("Trajectory shape:", vehicle["states"].shape)

In [ ]:
states = vehicle["states"]

plt.figure(figsize=(10, 8))

plt.plot(
    states[:, 0],
    states[:, 1],
    marker="o",
    markersize=2
)

plt.xlabel("X position")
plt.ylabel("Y position")
plt.title("OMEGA — Vehicle Trajectory")

plt.grid()
plt.show()

In [ ]:
MIN_HISTORY = 20
MIN_FUTURE = 60

usable_tracks = []

for track in trajectories:

    states = track["states"]

    if len(states) >= MIN_HISTORY + MIN_FUTURE:
        usable_tracks.append(track)

print("Total tracks:", len(trajectories))
print("Usable tracks:", len(usable_tracks))

In [ ]:
X = []
y = []

for track in usable_tracks:

    states = track["states"]

    past = states[:MIN_HISTORY]
    future = states[MIN_HISTORY:MIN_HISTORY + MIN_FUTURE]

    X.append(past)
    y.append(future)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_model = X[:, :, [0, 1, 5, 6]]
y_model = y[:, :, [0, 1, 5, 6]]

print("Input:", X_model.shape)
print("Target:", y_model.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_flat = X_model.reshape(-1, 4)

scaler.fit(X_flat)

X_scaled = scaler.transform(X_flat).reshape(X_model.shape)
y_scaled = scaler.transform(
    y_model.reshape(-1, 4)
).reshape(y_model.shape)

print("Scaled X:", X_scaled.shape)
print("Scaled y:", y_scaled.shape)

In [ ]:
import torch

X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y_scaled,
    dtype=torch.float32
)

print(X_tensor.shape)
print(y_tensor.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_tensor,
    y_tensor,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

In [ ]:
import torch.nn as nn

class TrajectoryLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=4,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(
            128,
            60 * 4
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        last_hidden = output[:, -1, :]

        prediction = self.fc(last_hidden)

        prediction = prediction.view(
            x.size(0),
            60,
            4
        )

        return prediction

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TrajectoryLSTM().to(device)

print(model)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        prediction = model(xb)

        loss = criterion(
            prediction,
            yb
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()

    val_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            prediction = model(xb)

            loss = criterion(prediction, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Train Loss: {train_loss:.5f} "
        f"Val Loss: {val_loss:.5f}"
    )

In [ ]:
from scipy.spatial.distance import cdist

scene_tracks = usable_tracks[:20]

print("Vehicles/agents:", len(scene_tracks))

In [ ]:
current_states = []

for track in scene_tracks:

    state = track["states"][MIN_HISTORY - 1]

    current_states.append([
        state[0],  # x
        state[1],  # y
        state[4],  # heading
        state[5],  # velocity x
        state[6]   # velocity y
    ])

current_states = np.array(current_states)

print("Scene state shape:", current_states.shape)

positions = current_states[:, :2]

distance_matrix = cdist(
    positions,
    positions
)

print(distance_matrix.shape)

In [ ]:
DISTANCE_THRESHOLD = 30.0

edges = []

for i in range(len(current_states)):

    for j in range(len(current_states)):

        if i == j:
            continue

        if distance_matrix[i, j] < DISTANCE_THRESHOLD:

            edges.append([i, j])

edges = np.array(edges)

print("Number of edges:", len(edges))

In [ ]:
import networkx as nx

graph = nx.DiGraph()

for i in range(len(current_states)):
    graph.add_node(i)

for source, target in edges:
    graph.add_edge(source, target)

plt.figure(figsize=(10, 8))

pos = {
    i: current_states[i, :2]
    for i in range(len(current_states))
}

nx.draw(
    graph,
    pos,
    with_labels=True,
    node_size=500
)

plt.title("OMEGA Dynamic Interaction Graph")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

In [ ]:
!pip install -q torch-geometric

In [ ]:
from torch_geometric.nn import GATConv

print("PyTorch Geometric loaded!")

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv


class SceneGAT(nn.Module):

    def __init__(self):

        super().__init__()

        self.gat1 = GATConv(
            in_channels=5,
            out_channels=32,
            heads=4
        )

        self.gat2 = GATConv(
            in_channels=32 * 4,
            out_channels=64,
            heads=2
        )

        self.output = nn.Linear(
            64 * 2,
            128
        )

    def forward(self, x, edge_index):

        x = self.gat1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.gat2(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.output(x)

        return x

In [ ]:
node_features = torch.tensor(
    current_states,
    dtype=torch.float32
).to(device)

edge_index = torch.tensor(
    edges.T,
    dtype=torch.long
).to(device)

print("Nodes:", node_features.shape)
print("Edges:", edge_index.shape)

gat = SceneGAT().to(device)

node_embeddings = gat(
    node_features,
    edge_index
)

print("Node embeddings:", node_embeddings.shape)

In [ ]:
class TemporalTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.input_projection = nn.Linear(
            4,
            128
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

    def forward(self, x):

        x = self.input_projection(x)

        x = self.transformer(x)

        return x[:, -1, :]

In [ ]:
class OMEGAFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.temporal = TemporalTransformer()

        self.gat = SceneGAT()

        self.fusion = nn.Linear(
            128 + 128,
            256
        )

        self.predictor = nn.Linear(
            256,
            60 * 4
        )

    def forward(
        self,
        trajectory,
        node_features,
        edge_index,
        target_index=0
    ):

        temporal_embedding = self.temporal(
            trajectory
        )

        node_embeddings = self.gat(
            node_features,
            edge_index
        )

        social_embedding = node_embeddings[
            target_index
        ]

        social_embedding = social_embedding.unsqueeze(0).expand(
            trajectory.size(0), -1
        )

        combined = torch.cat(
            [
                temporal_embedding,
                social_embedding
            ],
            dim=1
        )

        fused = torch.relu(
            self.fusion(combined)
        )

        prediction = self.predictor(
            fused
        )

        prediction = prediction.view(
            trajectory.size(0),
            60,
            4
        )

        return prediction